# Lesson 07 - Plánovací návrhový vzor

Tento notebook demonštruje **Plánovací návrhový vzor** pre AI agentov pomocou Microsoft Agent Framework.
Naučíte sa, ako rozložiť komplexné cestovné požiadavky na štruktúrované podúlohy, priradiť ich špecializovaným agentom
a vykonať výsledný plán — to všetko pomocou štruktúrovaného výstupu podporeného modelmi Pydantic.


## Nastavenie


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Rozklad úlohy

Rozklad úlohy je jadrom návrhového vzoru plánovania. Namiesto toho, aby sme požiadali jediného agenta, aby zvládol komplexnú požiadavku od začiatku do konca, rozdelíme problém na menšie, jasne definované **podúlohy**. Každá podúloha je pridelená špecializovanému agentovi (napr. lety, hotely, aktivity) s jasnými prioritami a poradím závislostí.

Tento prístup prináša niekoľko výhod:
- **Jasnosť**: každá podúloha má jednu zodpovednosť.
- **Paralelizmus**: nezávislé podúlohy môžu bežať súčasne.
- **Spoľahlivosť**: zlyhania sú izolované na jednotlivé podúlohy.
- **Sledovanie rozpočtu**: náklady sú odhadované pre každú podúlohu a zhrnuté.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Vytváranie plánovacieho agenta so štruktúrovaným výstupom

Plánovací agent vystupuje ako **koordinátor na recepcii**. Na základe vysokoúrovňovej cestovnej požiadavky vytvorí štruktúrovaný `TravelPlan` — rozloží požiadavku na podúlohy, nastaví priority a identifikuje závislosti tak, aby sprievodca alebo vrstva vykonávania mohli prácu realizovať.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Vykonávanie plánu so špecializovanými nástrojmi

Keď recepčný agent vytvorí štruktúrovaný plán, vykoná ho **konciérsky agent**. Každý špecializovaný nástroj spracováva jednu kategóriu podúlohy (letecké lety, hotely, aktivity). Konciér prechádza podúlohy plánu podľa poradia závislostí a priraďuje každú z nich príslušnému nástroju.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Zhrnutie

V tejto lekcii ste sa naučili **Plánovací vzor** pre AI agentov:

1. **Dezaggregácia úloh** — Agent plánujúci na recepcii rozdelí zložitú požiadavku na cestovanie na
   štruktúrované podúlohy pomocou Pydantic modelov, pričom každú priradí špecializovanému agentovi spolu s prioritami
   a závislosťami.
2. **Štruktúrovaný výstup** — Agent pri prenose `response_format` vráti validovaný
   objekt `TravelPlan` namiesto voľného textu, čo zabezpečuje spoľahlivé spracovanie ďalej.
3. **Realizácia plánu** — Concierge agent prechádza podúlohami pomocou špecializovaných nástrojov
   (`book_flight`, `reserve_hotel`, `book_activity`), aby vykonal plán a nahlásil výsledky.

Tento vzor oddeluje *čo robiť* (plánovanie) od *ako to robiť* (realizácia), čím sú agenti
modulárnejší, testovateľnejší a ľahšie rozšíriteľní.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Upozornenie**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, prosím, majte na pamäti, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nenesieme zodpovednosť za akékoľvek nedorozumenia alebo nesprávne výklady vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
